# Unidad: Redes Neuronales Artificiales
## PyTorch nativo y Keras 3 con backend PyTorch — Clasificación MNIST
### Inteligencia Artificial — Lic. en Sistemas — FCAD/UNER

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CristianPacifico/ia-ls-fcad-uner/blob/main/notebooks/ml/ann/03_ANN_PyTorch_Keras.ipynb)

## 🎯 Objetivos de Aprendizaje

Al finalizar este notebook vas a poder:

✅ Entender las diferencias clave entre **PyTorch nativo** y la API de alto nivel **Keras**  
✅ Construir y entrenar un MLP en **PyTorch** con el bucle de entrenamiento manual  
✅ Usar **Keras 3** con el backend PyTorch — misma API, motor PyTorch bajo el capó  
✅ Comparar la verbosidad, flexibilidad y resultados de ambos enfoques  

---

## 📖 Marco Teórico

### Ecosistema de Frameworks

Existen tres formas principales de construir redes neuronales en Python:

| Framework | Nivel | Control | Curva de aprendizaje |
|-----------|-------|---------|----------------------|
| **PyTorch nativo** | Bajo | Total (loop manual) | Alta |
| **TensorFlow + Keras** | Alto | Limitado | Baja |
| **Keras 3 multi-backend** | Alto | Limitado | Baja |

### Keras 3 — Multi-backend

Keras 3 (a partir de la versión 3.0) se separó de TensorFlow y ahora soporta **tres backends**:

- `tensorflow` (por defecto)
- `torch` ← lo que usaremos en Parte 2
- `jax`

```python
import os
os.environ["KERAS_BACKEND"] = "torch"  # antes de importar keras
import keras  # ahora usa PyTorch internamente
```

> ⚠️ La variable de entorno **debe establecerse antes** de importar `keras`. En un notebook, esto significa hacerlo en la primera celda.

### PyTorch — Loop manual de entrenamiento

En PyTorch nativo el proceso de entrenamiento se controla completamente:

```python
for epoch in range(EPOCHS):
    for X_batch, y_batch in dataloader:
        pred = model(X_batch)          # forward pass
        loss = loss_fn(pred, y_batch)  # calcular pérdida
        optimizer.zero_grad()          # limpiar gradientes
        loss.backward()                # backward pass (autograd)
        optimizer.step()               # actualizar pesos
```

Esto da más **flexibilidad** (p. ej. GAN, meta-learning) a cambio de más código.

### Diferencias conceptuales

| Aspecto | PyTorch nativo | Keras (cualquier backend) |
|---------|----------------|---------------------------|
| Definición del modelo | `nn.Module` + `forward()` | `Sequential` o API funcional |
| Entrenamiento | Loop manual | `model.fit()` |
| Evaluación | Loop manual | `model.evaluate()` |
| Predicciones | `model(X)` con `torch.no_grad()` | `model.predict(X)` |
| Historial de métricas | Dict manual | `history.history` automático |

## ⚙️ Configuración inicial

> ⚠️ **Importante**: `KERAS_BACKEND` debe establecerse **antes de importar keras**. Esta celda debe ser la primera en ejecutarse.

In [ ]:
import os
os.environ['KERAS_BACKEND'] = 'torch'   # Keras 3 usará PyTorch como motor
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

print("KERAS_BACKEND =", os.environ['KERAS_BACKEND'])

---
## 🔥 Parte 1: PyTorch Nativo

Vamos a construir y entrenar un MLP sobre MNIST usando la API de bajo nivel de PyTorch:
`nn.Module`, `DataLoader` y el loop de entrenamiento manual.

### 📦 Paso 1.1: Imports y dispositivo

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = (
    'cuda' if torch.cuda.is_available()
    else 'mps' if torch.backends.mps.is_available()
    else 'cpu'
)

print(f'PyTorch: {torch.__version__}')
print(f'Dispositivo: {device}')

### 🔢 Paso 1.2: Carga de datos con `DataLoader`

`torchvision.datasets.MNIST` descarga el dataset y aplica transformaciones en el pipeline.
`DataLoader` lo envuelve en un iterador de mini-batches.

La normalización con media `0.1307` y desviación estándar `0.3081` son los valores calculados sobre todo el dataset MNIST.

In [ ]:
BATCH_SIZE = 128

# Transformación: tensor + normalización con media/std de MNIST
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(root='./data', train=True,  download=True, transform=transform)
test_dataset  = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f'Train: {len(train_dataset):,} imágenes | {len(train_loader)} batches de {BATCH_SIZE}')
print(f'Test:  {len(test_dataset):,} imágenes  | {len(test_loader)} batches de {BATCH_SIZE}')

# Dimensiones de un batch
X_batch, y_batch = next(iter(train_loader))
print(f'Batch: X={X_batch.shape}, y={y_batch.shape}  | dtype={X_batch.dtype}')

In [ ]:
fig, axes = plt.subplots(2, 10, figsize=(14, 3))
for i in range(20):
    ax = axes[i // 10, i % 10]
    ax.imshow(X_batch[i].squeeze(), cmap='gray')
    ax.set_title(f'[{y_batch[i].item()}]', fontsize=9)
    ax.axis('off')
plt.suptitle('Muestras MNIST — DataLoader PyTorch', y=1.02)
plt.tight_layout()
plt.show()

### 🏗️ Paso 1.3: Definición del modelo con `nn.Module`

En PyTorch, toda red neuronal se define como una clase que hereda de `nn.Module`:

- `__init__`: define las capas como atributos
- `forward`: define el flujo de datos (forward pass)

El backward pass (**backpropagation**) lo realiza `autograd` automáticamente al llamar `.backward()` sobre la loss.

In [ ]:
class MLP(nn.Module):
    """Perceptrón Multicapa: 784 → 512 → 256 → 10"""

    def __init__(self, input_dim=784, hidden1=512, hidden2=256,
                 num_classes=10, dropout=0.2):
        super().__init__()
        self.network = nn.Sequential(
            nn.Flatten(),
            nn.Linear(input_dim, hidden1),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden1, hidden2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden2, num_classes)
            # ← Sin softmax: CrossEntropyLoss lo incluye internamente
        )

    def forward(self, x):
        return self.network(x)


model_pt = MLP().to(device)
print(model_pt)

total_params = sum(p.numel() for p in model_pt.parameters() if p.requires_grad)
print(f'\nParámetros entrenables: {total_params:,}')

### 🏋️ Paso 1.4: Loop de entrenamiento manual

En PyTorch el entrenamiento requiere implementar explícitamente:

1. `model.train()` / `model.eval()` — activa/desactiva Dropout y BatchNorm
2. `optimizer.zero_grad()` — limpia los gradientes acumulados del paso anterior
3. `loss.backward()` — calcula gradientes (backprop automático)
4. `optimizer.step()` — actualiza los pesos con los gradientes
5. `torch.no_grad()` — desactiva el grafo de cómputo durante evaluación

> 📌 `nn.CrossEntropyLoss` espera **logits** (sin softmax), e internamente aplica log-softmax + NLLLoss.

In [ ]:
def train_epoch(model, loader, loss_fn, optimizer, device):
    model.train()
    total_loss, correct = 0.0, 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        pred = model(X)
        loss = loss_fn(pred, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * X.size(0)
        correct += (pred.argmax(1) == y).sum().item()
    n = len(loader.dataset)
    return total_loss / n, correct / n


def eval_epoch(model, loader, loss_fn, device):
    model.eval()
    total_loss, correct = 0.0, 0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            total_loss += loss_fn(pred, y).item() * X.size(0)
            correct += (pred.argmax(1) == y).sum().item()
    n = len(loader.dataset)
    return total_loss / n, correct / n


print('Funciones de entrenamiento definidas ✅')

In [ ]:
EPOCHS = 15

loss_fn   = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_pt.parameters(), lr=1e-3)

history_pt = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = train_epoch(model_pt, train_loader, loss_fn, optimizer, device)
    va_loss, va_acc = eval_epoch(model_pt, test_loader,  loss_fn, device)

    history_pt['train_loss'].append(tr_loss)
    history_pt['val_loss'].append(va_loss)
    history_pt['train_acc'].append(tr_acc)
    history_pt['val_acc'].append(va_acc)

    if epoch == 1 or epoch % 5 == 0:
        print(f'Época {epoch:2d}/{EPOCHS} | '
              f'Loss: {tr_loss:.4f}  Acc: {tr_acc:.4f} | '
              f'Val Loss: {va_loss:.4f}  Val Acc: {va_acc:.4f}')

print(f'\n✅ PyTorch — Test Accuracy final: {history_pt["val_acc"][-1]*100:.2f}%')

### 📉 Paso 1.5: Curvas de entrenamiento

In [ ]:
epochs_range = range(1, EPOCHS + 1)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(epochs_range, history_pt['train_loss'], label='Entrenamiento', linewidth=2)
axes[0].plot(epochs_range, history_pt['val_loss'],   label='Test',          linewidth=2, linestyle='--')
axes[0].set_title('Pérdida (CrossEntropy)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(epochs_range, history_pt['train_acc'], label='Entrenamiento', linewidth=2)
axes[1].plot(epochs_range, history_pt['val_acc'],   label='Test',          linewidth=2, linestyle='--')
axes[1].set_title('Exactitud (Accuracy)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Época')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle('Curvas de entrenamiento — PyTorch nativo', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 📊 Paso 1.6: Evaluación final y matriz de confusión

In [ ]:
model_pt.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for X, y in test_loader:
        X = X.to(device)
        logits = model_pt(X)
        preds = logits.argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(y.numpy())

y_pred_pt = np.array(all_preds)
y_true_pt = np.array(all_labels)

cm_pt = confusion_matrix(y_true_pt, y_pred_pt)

plt.figure(figsize=(9, 7))
sns.heatmap(cm_pt, annot=True, fmt='d', cmap='Blues',
            xticklabels=range(10), yticklabels=range(10))
plt.xlabel('Predicho',  fontsize=12)
plt.ylabel('Real',      fontsize=12)
plt.title('Matriz de Confusión — PyTorch nativo', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(classification_report(y_true_pt, y_pred_pt, target_names=[str(i) for i in range(10)]))

---
## 🟠 Parte 2: Keras 3 con backend PyTorch

Ahora usaremos la **misma API de Keras** que en el notebook anterior, pero con un cambio clave:  
el motor de cómputo es **PyTorch** (establecido con `KERAS_BACKEND=torch` al inicio).

El modelo resultante es un objeto PyTorch — Keras simplemente actúa como capa de abstracción.

### 📦 Paso 2.1: Importar Keras 3

In [ ]:
import keras
from keras.models import Sequential
from keras.layers import Dense, Dropout, Input
from keras.utils import to_categorical
from keras.datasets import mnist as keras_mnist

keras.utils.set_random_seed(SEED)

print(f'Keras:   {keras.__version__}')
print(f'Backend: {keras.backend.backend()}')   # debe decir "torch"

### 🔧 Paso 2.2: Preprocesamiento

Keras espera **arrays NumPy** normalizados a `[0, 1]` con etiquetas en one-hot.  
Usamos `keras.datasets.mnist` para consistencia (mismos datos que torchvision).

In [ ]:
(X_train_k, y_train_k), (X_test_k, y_test_k) = keras_mnist.load_data()

# Aplanar y normalizar
X_train_k = X_train_k.reshape(-1, 784).astype('float32') / 255.0
X_test_k  = X_test_k.reshape(-1, 784).astype('float32')  / 255.0

# One-hot encoding
NUM_CLASSES = 10
y_train_cat = to_categorical(y_train_k, NUM_CLASSES)
y_test_cat  = to_categorical(y_test_k,  NUM_CLASSES)

print(f'X_train: {X_train_k.shape} | rango [{X_train_k.min():.1f}, {X_train_k.max():.1f}]')
print(f'y_train (one-hot): {y_train_cat.shape}')

### 🏗️ Paso 2.3: Modelo Sequential — misma API, backend PyTorch

La arquitectura es idéntica a la del notebook anterior (`02_ANN_Keras_MNIST`).  
La diferencia es que internamente Keras 3 crea módulos **PyTorch** (`torch.nn.Module`).

In [ ]:
model_k = Sequential([
    Input(shape=(784,)),
    Dense(512, activation='relu',    name='capa_entrada'),
    Dropout(0.2,                     name='dropout_1'),
    Dense(256, activation='relu',    name='capa_oculta'),
    Dropout(0.2,                     name='dropout_2'),
    Dense(NUM_CLASSES, activation='softmax', name='capa_salida')
], name='ANN_MNIST_torch_backend')

model_k.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model_k.summary()

# Verificar que internamente es un módulo PyTorch
print(f'\n¿Es un nn.Module de PyTorch? {isinstance(model_k, torch.nn.Module)}')

### 🏋️ Paso 2.4: Entrenamiento con `model.fit()`

La API de Keras encapsula todo el loop: forward, backward, actualización y métricas.

In [ ]:
history_k = model_k.fit(
    X_train_k, y_train_cat,
    epochs           = EPOCHS,
    batch_size       = BATCH_SIZE,
    validation_split = 0.1,
    verbose          = 1
)

print(f'\n✅ Keras (backend=torch) — entrenamiento completo ({EPOCHS} épocas)')

### 📉 Paso 2.5: Curvas de entrenamiento

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(history_k.history['loss'],     label='Entrenamiento', linewidth=2)
axes[0].plot(history_k.history['val_loss'], label='Validación',    linewidth=2, linestyle='--')
axes[0].set_title('Pérdida (Loss)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Categorical Crossentropy')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(history_k.history['accuracy'],     label='Entrenamiento', linewidth=2)
axes[1].plot(history_k.history['val_accuracy'], label='Validación',    linewidth=2, linestyle='--')
axes[1].set_title('Exactitud (Accuracy)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Época')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle('Curvas de entrenamiento — Keras 3 (backend=torch)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 📊 Paso 2.6: Evaluación y matriz de confusión

In [ ]:
test_loss_k, test_acc_k = model_k.evaluate(X_test_k, y_test_cat, verbose=0)
print(f'Keras + PyTorch | Test Loss: {test_loss_k:.4f} | Test Accuracy: {test_acc_k*100:.2f}%')

y_pred_k = np.argmax(model_k.predict(X_test_k, verbose=0), axis=1)

cm_k = confusion_matrix(y_test_k, y_pred_k)

plt.figure(figsize=(9, 7))
sns.heatmap(cm_k, annot=True, fmt='d', cmap='Oranges',
            xticklabels=range(10), yticklabels=range(10))
plt.xlabel('Predicho',  fontsize=12)
plt.ylabel('Real',      fontsize=12)
plt.title('Matriz de Confusión — Keras 3 (backend=torch)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 🔍 Parte 3: Comparación lado a lado

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
axes[0].plot(epochs_range, history_pt['val_acc'],
             label='PyTorch nativo', linewidth=2, color='steelblue')
axes[0].plot(range(1, len(history_k.history['val_accuracy']) + 1),
             history_k.history['val_accuracy'],
             label='Keras (backend=torch)', linewidth=2, linestyle='--', color='darkorange')
axes[0].set_title('Accuracy en Validación / Test', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Loss
axes[1].plot(epochs_range, history_pt['val_loss'],
             label='PyTorch nativo', linewidth=2, color='steelblue')
axes[1].plot(range(1, len(history_k.history['val_loss']) + 1),
             history_k.history['val_loss'],
             label='Keras (backend=torch)', linewidth=2, linestyle='--', color='darkorange')
axes[1].set_title('Loss en Validación / Test', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Época')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle('Comparación: PyTorch nativo vs Keras 3 (backend=torch)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

acc_pt = history_pt['val_acc'][-1] * 100
acc_k  = test_acc_k * 100
print(f'Test Accuracy | PyTorch nativo: {acc_pt:.2f}% | Keras+torch: {acc_k:.2f}%')

### Resumen comparativo

| Característica | PyTorch Nativo | Keras 3 (backend=torch) |
|----------------|----------------|-------------------------|
| **Líneas de código** | ~60 (loop + fns) | ~15 (compile + fit) |
| **Control del loop** | Total | Delegado a Keras |
| **Métricas automáticas** | Manual | Automáticas |
| **Motor de cómputo** | PyTorch | PyTorch |
| **Accuracy esperado** | ~98% | ~98% |
| **Curva de aprendizaje** | Alta | Baja |
| **Flexibilidad** | Máxima | Moderada |
| **Uso recomendado** | Investigación, arquitecturas custom | Prototipado rápido, enseñanza |

> 💡 **Cuándo usar cada uno:**  
> - **PyTorch nativo**: cuando necesitás control total sobre el loop (GAN, RL, meta-learning, pérdidas custom).  
> - **Keras**: cuando la arquitectura es estándar y querés iterar rápido.  
> - **Keras + backend=torch**: lo mejor de ambos — portabilidad de Keras + ecosistema PyTorch (`.pth`, ONNX, etc.).

---
## 🎓 Conclusiones

En este notebook comparamos dos formas de entrenar el **mismo modelo** (MLP 784→512→256→10) sobre **MNIST**:

### PyTorch nativo
- Define el modelo como una clase que hereda de `nn.Module`
- Requiere implementar el **loop de entrenamiento** manualmente
- Máxima flexibilidad y control
- Estándar en investigación de deep learning

### Keras 3 con backend PyTorch
- Misma API de alto nivel (`Sequential`, `compile`, `fit`) que en el notebook anterior
- El backend se cambia con **una sola línea**: `os.environ['KERAS_BACKEND'] = 'torch'`
- El modelo resultante **es un `nn.Module`** de PyTorch
- Permite combinar la comodidad de Keras con el ecosistema de PyTorch

### Próximos pasos
- Explorar **arquitecturas convolucionales (CNN)** para aprovechar la estructura espacial de las imágenes
- Implementar **callbacks** en Keras (EarlyStopping, ModelCheckpoint)
- Probar **loops de entrenamiento custom** en Keras con `train_step`

---
> 💡 Keras 3 multi-backend es una herramienta poderosa para escribir código portable: el mismo modelo puede correr sobre TensorFlow, PyTorch o JAX sin modificar la arquitectura.

---
<br>

**Inteligencia Artificial** · Licenciatura en Sistemas · FCAD - UNER  
Autor: Cristian Pacífico · [GitHub](https://github.com/CristianPacifico/ia-ls-fcad-uner)  
Licencia: [MIT](https://opensource.org/licenses/MIT)